# 5.4 Het antwoord: een verhouding die kantelt

[05.3](05.3-covid-modelling%20jeffrey.ipynb) liet een rechte lijn achter in positieve tests met een residu dat na januari 2021 negatief duikt en nooit meer terugkeert. Dit notebook is het uitgewerkte antwoord, en het antwoord is compact: **dezelfde lijn, vermenigvuldigd met een schakelaar (switch).**

    deaths_shifted ≈ (a · positivetests + b) · logistic(day, k, x0)

De lijn is de claim dat overlijdens een vaste fractie zijn van positieve tests. De switch is de claim dat de fractie *kantelt* — één waarde, een overgang over enkele weken, een lagere waarde. Twee parameters beschrijven de switch: `x0`, de dag waarop de omslag halverwege is, en `k`, hoe steil hij verloopt (negatief, zodat hij daalt van 1 richting 0). Vier parameters in totaal, en elk daarvan betekent iets wat je achteraf direct kunt interpreteren.

Vermenigvuldigd, niet opgeteld. Een logistische functie *opgeteld* bij de lijn zou beweren dat "vanaf een zekere dag een vast aantal doden per dag verdwijnt, ongeacht hoeveel mensen positief testen". Dat is niet de claim. Een beschermde populatie verandert de *fractie* van positieve tests die in overlijden eindigt, en een fractie grijpt aan op de omvang van de tests: hoe meer tests, hoe meer overlijdens de switch afroomt. Een verhouding is een vermenigvuldiging, dus de term wordt vermenigvuldigd. Dat correct opschrijven is de essentie van het modelleren; de rest is `train_model`.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.analytics import DistributionFitter, fit_table
from goad_toolkit.config import DataConfig, FileConfig
from goad_toolkit.dataprocessor import CovidDataProcessor
from goad_toolkit.models import linear_model, logistic, mse, train_model
from goad_toolkit.visualizer import (
    ComparePlot,
    ComparePlotDate,
    FitPlotSettings,
    PlotFits,
    PlotSettings,
    ResidualPlot,
)
from wa_analyzer.data import load_showcase

from scripts.covid_pipeline import fit_model, plot_residual, preprocess

## 5.4.1 De schakelaar, en waar te beginnen

`goad_toolkit.models.logistic(x, k, x0)` is vlak, dan een bocht, en dan weer vlak. `x0` plaatst de omslag en `k` bepaalt hoe snel hij plaatsvindt; `limit` blijft 1, omdat de switch de lijn schaalt in plaats van vervangt.

In [ ]:
t = np.linspace(-10, 10, 200)
shapes = pd.DataFrame({"t": t, "k=1: up": logistic(t, k=1, x0=0), "k=-0.5: down, slower": logistic(t, k=-0.5, x0=0)})
fig, ax = ComparePlot(PlotSettings(figsize=(7, 3.2), xlabel="t", ylabel="", title="A logistic switch")).plot(  # ty: ignore[invalid-argument-type]
    data=shapes, x="t", y1="k=1: up", y2="k=-0.5: down, slower",
)

De data is het dataframe dat 05.3 heeft gebouwd. De lijn wordt eerst gefit op het venster vóór de vaccinatie, om dezelfde reden als daar — dat is het venster waarin de verhouding standhoudt — en omdat de parameters `a` en `b` daarvan de startwaarden vormen die de 4-parameter-fit nodig heeft. De andere twee startwaarden volgen uit wat je al weet: `k` licht negatief, `x0` een paar weken na de eerste vaccinaties. Een optimizer die mijlenver van het antwoord start, zal gerust op een ander optimum landen. De grenzen (bounds) bepalen wat elke parameter mag betekenen: een ratio tussen 0 en 1, een intercept dat niet negatief is, een switch die *omlaag* kantelt, en een omslagpunt dat binnen het tijdvenster valt.

In [ ]:
VACCINATION_START = "2021-01-06"
data = CovidDataProcessor(FileConfig(), DataConfig()).process()
before = data.index < VACCINATION_START

tests = data["positivetests"].to_numpy()
deaths = data["deaths_shifted"].to_numpy()
day = np.arange(len(data)).astype(float)
X = np.stack([tests, day], axis=1)

line = train_model(tests[before], deaths[before], linear_model, mse, [0.01, 1.0],
                   bounds=[(0, 1.0), (0, None)])
data["line"] = linear_model(tests, line)


def covid_model(X: np.ndarray, params: list[float]) -> np.ndarray:
    """A straight line in positive tests, times a logistic switch on the day."""
    a, b, k, x0 = params
    return linear_model(X[:, 0], [a, b]) * logistic(X[:, 1], k=k, x0=x0)


vaccination_day = float(np.argmax(data.index >= VACCINATION_START))
initial = [line[0], line[1], -0.1, vaccination_day + 30]
params = train_model(X, deaths, covid_model, mse, initial,
                     bounds=[(0, 1.0), (0, None), (-1.0, 0), (0, len(data))])
a, b, k, x0 = params

data["turning"] = covid_model(X, params)
data["residual"] = data["deaths_shifted"] - data["turning"]
halfway = data.index[int(round(x0))].date()
print(f"the line alone:  deaths ≈ {line[0]:.4f} · tests + {line[1]:.1f}")
print(f"with the switch: a = {a:.4f}, b = {b:.1f}, k = {k:.3f}, x0 = day {x0:.0f} = {halfway}")
print(f"mse: the line on the full window {mse(deaths, data['line']):.0f}  →  the line times the switch {mse(deaths, data['turning']):.0f}")

In [ ]:
fig, ax = ComparePlotDate(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="deaths",
                                       title="Deaths, and a model whose ratio turns")).plot(
    data=data, x="date", y1="deaths_shifted", y2="turning",
    date=VACCINATION_START, datelabel="vaccination starts",
)

## 5.4.2 De parameters aflezen

`x0` plaatst het middelpunt van de omslag in **half maart 2021**, ongeveer tien weken na de eerste vaccinaties. `k` is klein, wat betekent dat de omslag geleidelijk verloopt: de schakelaar daalt van ongeveer vier vijfde naar één vijfde over de twee maanden gecentreerd rond die datum, en tegen mei is de verhouding nog maar iets meer dan een tiende van de winterwaarde.

Lees `a` en `b` met meer zorg. Het zijn niet de getallen van de pre-vaccinatielijn — de helling viel lager uit en het intercept hoger — omdat een dermate geleidelijke switch in januari en februari al iets onder de 1 zit, waardoor de lijn zich aanpast om het werk ermee te verdelen. Vier parameters die gezamenlijk zijn gefit, zijn geen vier afzonderlijke bevindingen; ze geïsoleerd interpreteren is een denkfout. Wat het model op een bepaalde dag beweert is hun *product*, de ratio van overlijdens tot tests die het impliceert, en dat is het getal om te citeren.

Zet dat af tegen de uitrol. Het Nederlandse programma startte op 6 januari met verpleeghuisbewoners en hun zorgpersoneel, en werkte gedurende februari en maart stapsgewijs omlaag door de leeftijdsgroepen vanaf de oudsten — wat wil zeggen dat het de mensen bereikte die het leeuwendeel van de sterfgevallen droegen lang vóórdat het de meerderheid van de bevolking bereikte. Een omslag in de overlijdens-per-test-ratio die half maart al op de helft is, terwijl de algehele vaccinatiegraad nog laag was, is exact wat die volgorde voorspelt. De parameters zijn een ontdekking: ze dateren de omslag, en die datum strookt met het mechanisme dat je voor ogen had.

Strookt met, geen bewijs van. Het model heeft een schakelaar op de kalender, geen term voor vaccinatie zelf. Het einde van de wintergolf, een verschuiving in wie zich liet testen, en de volgorde waarin leeftijden werden gevaccineerd liggen allemaal op dezelfde kalender, en elk daarvan zou een switch produceren die past. Het fitten van het residu toonde dát de ratio kantelde, en wanneer. **Waarom zij kantelde is een afzonderlijke claim** — de mechanismepijler uit [05.1](05.1-relationships%20jeffrey.ipynb), die door geen enkele fit wordt geleverd — en het verdedigen daarvan vereist iets buiten dit model: vaccinatiegraad per leeftijdsgroep afgezet tegen sterfte per leeftijdsgroep, bijvoorbeeld, of dezelfde fit in een land waar de uitrol op een andere datum begon.

In [ ]:
fig, ax = ResidualPlot(PlotSettings(figsize=(11, 4), xlabel="date", ylabel="error",
                                    title="The residual, once the ratio is allowed to turn")).plot(
    data=data, x="date", y="residual", date=VACCINATION_START,
    datelabel="vaccination starts", interval=1,
)

In [ ]:
fitter = DistributionFitter(seed=42)
fits = fitter.fit(data["residual"].to_numpy(), discrete=False)
print(fit_table(fits)[["distribution", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]].head(4).to_string())

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="error", ylabel="density",
                            title="Full window, with the switch: the residual, top three fits")).plot(
    data=data["residual"].to_numpy(), fit_results=fits,
    fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

De lange negatieve reeks is verdwenen, de fout is nog maar een tiende van wat de rechte lijn achterliet, en het residu lijkt weer op dat van vóór de vaccinatie: symmetrisch, meerdere families binnen enkele log-likelihood punten van elkaar, geen enkele verworpen. Langs de lat van [05.3](05.3-covid-modelling%20jeffrey.ipynb) — geen structuur in de tijd, een plausibele familie — is dit model klaar. Niet perfect: de wekelijkse golven zijn er nog steeds, en een serieus model zou die meenemen (een sinus met periode 7, of het weekcyclus-effect eruit gefilterd in de pipeline). Maar er blijft niets over in het residu dat een vorm heeft die wijst op een mechanisme dat het model mist.

Twee dingen vallen op aan hoe compact de oplossing was. Er werd één term en twee parameters toegevoegd, en de lijn zelf bleef onaangetast. En de term werd gekozen door te vragen wat de data betekent — een fractie van tests, die over weken verandert — in plaats van vormen uit te proberen tot er eentje paste. Dat is het verschil tussen een model dat je inhoudelijk kunt verdedigen en eentje dat simpelweg toevallig aansluit.

## 5.4.3 Nog één basisfunctie

De switch is er eentje uit een kleine familie van vormen die de moeite waard zijn om bij naam te kennen — **lineair, sinus, exponentieel, logistisch** — en de gewoonte is voor elk hetzelfde: kijk naar wat het huidige model niet kan verklaren, benoem de vorm van wat overblijft, en voeg die vorm toe als term. Een lijn zegt "evenredig". Een exponentieel verband zegt "samengestelde groei". Een logistische functie zegt "het kantelt". Een sinus zegt "het herhaalt zich". Maandelijkse vliegtuigpassagiers (1949–1960) is het duidelijkste voorbeeld van die laatste: een trend, en een cyclus die elke twaalf maanden terugkeert.

In [ ]:
flights = load_showcase("flights")
months = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"], format="%Y-%b")
t = np.arange(len(flights)).astype(float)
y = flights["passengers"].to_numpy().astype(float)


def trend_plus_seasonal(t: np.ndarray, params: list[float]) -> np.ndarray:
    a, b, amplitude, phase = params
    return a * t + b + amplitude * np.sin(2 * np.pi * t / 12 + phase)


trend_only = train_model(t, y, linear_model, mse, [1.0, 100.0])
trend_and_cycle = train_model(t, y, trend_plus_seasonal, mse, [1.0, 100.0, 30.0, 0.0])
compare = pd.DataFrame({
    "month": months,
    "actual": y,
    "trend only": linear_model(t, trend_only),
    "trend + 12-month sine": trend_plus_seasonal(t, trend_and_cycle),
})
for column in ["trend only", "trend + 12-month sine"]:
    explained = 1 - np.var(y - compare[column]) / np.var(y)
    print(f"{column:22s} variance explained {explained:.1%}")

settings = PlotSettings(figsize=(10, 4.5), title="Airline passengers: a line, and a line plus a sine",  # ty: ignore[invalid-argument-type]
                        xlabel="", ylabel="passengers")
fig, ax = ComparePlot(settings).plot(data=compare, x="month", y1="trend only", y2="trend + 12-month sine")
ax.scatter(compare["month"], compare["actual"], s=8, color="black", alpha=0.5, label="actual", zorder=3)
_ = ax.legend()

Eén toegevoegde term — de amplitude en fase van een 12-maandelijkse sinus, vier parameters in totaal — tilt de verklaarde variantie op van 85% naar 93%. Het gat dat overblijft is zichtbaar en eerlijk: de werkelijke cyclus *wordt breder* naarmate de trend groeit, en een sinus die bovenop een lijn wordt opgeteld blijft overal even breed. Om de rest te vangen zou je de cyclus moeten vermenigvuldigen met de trend in plaats van optellen — exact dezelfde opgeteld-of-vermenigvuldigd beslissing waarop de covid-switch draaide, en de logische volgende term in plaats van een falen van deze.

## 5.4.4 Waarom dit in een script leeft

De bovenstaande cyclus — verwerken, vergelijken, modelleren, residu, verdelingsfit — is ook `scripts/covid_pipeline.py`: zes functies en een `main()` die ze op volgorde uitvoert. Open het naast dit notebook. Elke functie heeft een naam waarvan een lezer de inhoud kan raden, en een signatuur die aangeeft wat erin gaat en wat eruit komt:

| functie | in | uit | de beslissing die het beheert |
|---|---|---|---|
| `preprocess()` | niets (config) | verwerkt dataframe | welke pipeline-stappen de data maken |
| `plot_zscores(data)` | dataframe | een figuur | of de twee reeksen dezelfde vorm delen |
| `fit_model(data)` | dataframe | dataframe, plus `predicted deaths` en `residual` | het model, startwaarden, bounds |
| `plot_model(data)` | gefit dataframe | een figuur | hoe de fit eruitziet tegen de data |
| `plot_residual(data)` | gefit dataframe | een figuur | of de fout structuur in de tijd vertoont |
| `plot_residual_distribution(data)` | gefit dataframe | figuur en `fit_table` | of de fout zuivere ruis is |

Geen enkele functie is langer dan twintig regels, en geen enkele weet van de anderen af: `fit_model` retourneert een *nieuw* dataframe in plaats van het bestaande aan te passen, waardoor tweemaal draaien niet leidt tot een dubbele fit, en elke plotfunctie accepteert het dataframe dat hij nodig heeft als argument in plaats van naar een globale variabele te grijpen. Een notebookcel heeft zo'n scheiding niet — niets houdt een variabele van de ene cel tegen om door te lekken naar de volgende, en niets dwingt de logica compact genoeg te houden om te testen.

Drie dingen die het script kan en de cellen hierboven niet: één stap opnieuw draaien (pas het model aan in `fit_model`, en niets bovenstrooms hoeft opnieuw te draaien); geïmporteerd worden — de volgende cel doet exact dat, en een test zou `fit_model` kunnen aanroepen op tien gefingeerde rijen; en draaien op een planning (`uv run python scripts/covid_pipeline.py`), zonder kernel, zonder celvolgorde, en zonder menselijke tussenkomst. De prijs is dat een functie expliciet moet *zeggen* wat hij nodig heeft. Die beperking is juist de kracht.

In [ ]:
scripted = fit_model(preprocess())
fig, ax = plot_residual(scripted)

Dezelfde fit en hetzelfde residu als in §5.4.2, in twee functieaanroepen; de logregels erboven zijn het script dat rapporteert wat het heeft geladen, gefit en waar het de figuur heeft opgeslagen.

## Wat mee te nemen

1. **Het residu is de bevinding, niet het overschot.** De lijn werd verbeterd door te vragen hoe de *fout* er nog uitzag, niet door langer naar de fit te staren; de vorm van de fout benoemde de term, en de dag waarop hij begon dateerde hem.
2. **Een basisfunctie is een claim over de vorm die je verwacht.** Een logistische functie stelde "de ratio kantelt". Een sinus stelde "dit herhaalt zich elke twaalf maanden". Beide zijn toetsbaar, en beide werden bevestigd doordat het residu kromp zodra ze werden toegevoegd — niet aangenomen omdat ze toevallig aannemelijk leken. Hoe de term combineert met wat er al staat, is onderdeel van de claim.
3. **Het model dateert de omslag; het mechanisme is aan jou om te verdedigen.** Een schakelaar op de kalender strookt met vaccinatie, én met al het andere dat op die kalender plaatsvond.
4. **Een script is een notebook waaruit het lekken is verwijderd.** Kleine functies met expliciete inputs stellen je in staat om één stap opnieuw te draaien, deze elders te importeren, of te testen zonder alles erboven opnieuw te hoeven draaien.